
# Intelligent Insurance Support Assistant
## Build Specification and Implementation Guide

### Objective
Build a modular insurance-support application using LangChain and Gemini. It should accept a customer query, identify its intent, summarize the issue, suggest practical next steps, and generate a customer-friendly response.

Demonstrate prompt templates, chat models, output parsing, structured output, Runnables, LCEL, parallel execution, batching, streaming, and basic error handling.

**Scope:** Do not use LangGraph, RAG, embeddings, or a vector database in this implementation. The focus is on composing LangChain components with LCEL.

## 1. Business context
An insurance company receives customer queries about claims, policy information, premium payments, renewals, deductibles, and general insurance questions. The initial support process involves identifying each query's category, summarizing the issue, and suggesting suitable next steps.

The assistant must not invent policy-specific coverage, claim eligibility, or approval decisions. It can explain general processes and direct customers to the insurer for policy-specific confirmation.

## 2. Functional requirements

### 2.1 Model configuration
- Initialize Gemini through LangChain.
- Read the API key from environment variables.
- Keep model configuration separate from application logic.
- Use `init_chat_model()` or the provider-specific Gemini integration.
- Keep the model identifier configurable.

### 2.2 Prompt templates
Create reusable prompts for:
1. The assistant's role and response behaviour.
2. Customer-query classification.
3. Customer-query summarization.
4. Recommended next steps.
5. Consolidating the analysis into a customer-friendly response.

Use `ChatPromptTemplate` where structured chat messages and roles are useful. Use `PromptTemplate` where a simple text prompt is sufficient.

### 2.3 Query classification
Classify each query into exactly one category:
- `Claim`
- `Policy`
- `Premium`
- `Renewal`
- `Other`

Return structured output with:
- `category`: one of the five allowed categories.
- `confidence`: model-estimated confidence between 0 and 1.
- `reason`: short explanation for the classification.

Example input:

> I met with an accident yesterday, and my car has significant damage. How can I claim insurance?

Illustrative output:

```json
{
  "category": "Claim",
  "confidence": 0.95,
  "reason": "The customer is asking about initiating an insurance claim."
}
```

Use a Pydantic schema with the model's structured-output capability. Treat confidence as a model estimate, not a calibrated probability.

### 2.4 Query summary
Generate a concise summary of the customer's query in two or three sentences.

Example input:

> I paid my premium last month, but I haven't received any confirmation. What should I do?

Illustrative output:

> The customer has paid their insurance premium but has not received payment confirmation. They need assistance verifying the payment status.

Use an appropriate output parser.

### 2.5 Recommended next steps
Generate three practical next steps based on the query.

Example input:

> I met with an accident yesterday, and my car has significant damage.

Illustrative output:
1. Ensure everyone's safety and contact emergency services if necessary.
2. Document the damage with photographs and collect relevant accident details.
3. Contact the insurer to understand the claim-registration process.

Do not invent policy-specific requirements or imply that a claim is guaranteed to be accepted.

### 2.6 Parallel analysis
Generate these three outputs independently from the same customer query:
- Query summary
- Query classification
- Recommended next steps

Use `RunnableParallel`. Return a dictionary with the keys `summary`, `classification`, and `recommended_steps`.

Illustrative result shape:

```python
{
    "summary": "...",
    "classification": {
        "category": "Claim",
        "confidence": 0.95,
        "reason": "..."
    },
    "recommended_steps": "..."
}
```

### 2.7 Consolidated customer response
Create a final prompt that receives the parallel analysis and generates a customer-friendly response containing:
- An acknowledgement of the customer's concern.
- A brief summary of the issue.
- Appropriate next steps.
- A professional closing statement.

Compose the components with LCEL.

### 2.8 Multiple-query processing
Support multiple queries through `batch()`.

| Scenario | Example query |
|---|---|
| Claim | I met with an accident yesterday. How do I initiate a claim? |
| Premium | I paid my premium but haven't received confirmation. |
| Renewal | My insurance policy expires next week. How can I renew it? |

### 2.9 Streaming
Demonstrate streaming for the consolidated response using `stream()`.

When displaying chunks in a terminal:
- `end=""` prevents `print()` from adding a newline after each chunk.
- `flush=True` asks Python to flush buffered output immediately.

`flush=True` affects output display buffering; it does not control Gemini generation or LangChain streaming.

### 2.10 Basic error handling
Handle:
- Empty customer query.
- Model/API execution failure.
- Unexpected model response.
- Invalid structured output.
- Failure in one of the parallel operations.

Do not silently turn a failed operation into a misleading customer-facing response.

## 3. LangChain concepts to demonstrate

| Concept | Expected use |
|---|---|
| `init_chat_model()` | Initialize Gemini through a common model interface |
| `ChatPromptTemplate` | Construct structured chat prompts |
| `MessagesPlaceholder` | Insert supplied conversation history |
| `StrOutputParser` | Convert an AI message into text |
| Structured output | Produce a validated Pydantic object |
| Runnable interface | Execute components through a common interface |
| `RunnableSequence` | Compose sequential operations |
| LCEL | Connect components using `\|` |
| `RunnableParallel` | Execute independent operations against the same input |
| `RunnableLambda` | Integrate ordinary Python functions into a pipeline |
| `RunnablePassthrough` | Preserve or forward input through a pipeline |
| `invoke()` | Process one input |
| `batch()` | Process multiple inputs |
| `stream()` | Consume output incrementally |
| Error handling | Handle model, parsing, and branch failures |

`RunnableLambda`, `RunnablePassthrough`, and conversation-history support can be added after the core pipeline works.

## 4. Suggested application structure

```text
insurance_support_assistant/
├── .env
├── .gitignore
├── environment.yml
├── app/
│   ├── __init__.py
│   ├── config.py
│   ├── main.py
│   ├── llm/
│   │   ├── __init__.py
│   │   └── model.py
│   ├── prompts/
│   │   ├── __init__.py
│   │   └── templates.py
│   ├── schemas/
│   │   ├── __init__.py
│   │   └── classification.py
│   └── services/
│       ├── __init__.py
│       └── assistant.py
└── experiments/
    └── langchain_basics.ipynb
```

### Suggested responsibilities

| File | Responsibility |
|---|---|
| `config.py` | Load environment variables and model settings |
| `llm/model.py` | Initialize and expose the configured chat model |
| `prompts/templates.py` | Define reusable prompt templates |
| `schemas/classification.py` | Define the classification schema |
| `services/assistant.py` | Compose classification, summary, recommendations, and response pipelines |
| `main.py` | Accept input and invoke the application service |

## 5. Example input scenarios

Use these queries to verify different intents. Responses must be generated dynamically; do not hardcode them.

**Scenario A — Claim**

> I was involved in a road accident yesterday. My car's front bumper and headlights are damaged. What should I do to start an insurance claim?

**Scenario B — Premium**

> I paid my insurance premium three days ago, but the payment is still not showing in my account. Can you help me understand what to do?

**Scenario C — Renewal**

> My vehicle insurance expires next week. How can I renew it, and what information should I keep ready?

**Scenario D — Policy**

> What does an insurance deductible mean, and how does it affect the amount I pay during a claim?

**Scenario E — Other**

> Can you explain the difference between third-party insurance and comprehensive insurance?

## 6. Implementation sequence

1. Configure the environment and Gemini model.
2. Define the Pydantic classification schema.
3. Create and test the prompt templates.
4. Build and test the classification chain.
5. Build and test the summary chain.
6. Build and test the next-steps chain.
7. Combine the three chains with `RunnableParallel`.
8. Create the consolidated-response chain.
9. Compose the complete pipeline with LCEL.
10. Test individual inputs with `invoke()`.
11. Test multiple inputs with `batch()`.
12. Demonstrate incremental output with `stream()`.
13. Add validation and error handling.
14. Refactor reusable logic into the suggested modules.
15. Add conversation-history support as an extension.

At each stage, inspect input and output types. Confirm that each Runnable's output is compatible with the next component's expected input.

## 7. Verification checklist

- [ ] Gemini API key is loaded from `.env` and is not committed to version control.
- [ ] Model initialization is centralized.
- [ ] Prompt templates are reusable and variables are supplied correctly.
- [ ] Classification returns the required structured fields.
- [ ] Classification category is restricted to the five allowed values.
- [ ] Summary is concise and reflects the supplied query.
- [ ] Recommended steps are relevant and do not invent policy-specific facts.
- [ ] `RunnableParallel` receives the same query in all three branches.
- [ ] Parallel results use the expected dictionary keys.
- [ ] The final response uses the parallel results.
- [ ] `invoke()` works for a single query.
- [ ] `batch()` works for multiple queries.
- [ ] `stream()` displays output incrementally.
- [ ] Empty input is handled clearly.
- [ ] Model and parsing failures are handled explicitly.
- [ ] Code is separated into reusable modules.
- [ ] No RAG, vector database, or LangGraph components are used.

## 8. Optional extension: conversation history

Extend the assistant to support follow-up questions within a session.

Example:

**Customer:** I want to know how to claim insurance after an accident.

**Assistant:** You can begin by registering a claim with your insurer. Keep the accident details and relevant documents available.

**Customer:** What documents will I need?

**Assistant:** For an accident-related claim, insurers commonly request policy details, a claim form, vehicle information, and evidence of the incident. Exact requirements depend on the insurer and policy.

Use `MessagesPlaceholder` to insert supplied conversation history into the prompt. Implement session-based history handling so one customer's conversation is not mixed with another customer's conversation.

Remember: `MessagesPlaceholder` inserts history into a prompt; it does not, by itself, store or manage that history.

## 9. Design and implementation principles

- Keep model configuration, prompts, schemas, and orchestration separate.
- Prefer reusable chains over repeated prompt/model/parser code.
- Use parallel execution only for independent operations.
- Use sequential composition when a later operation depends on an earlier result.
- Validate model-generated structured data before using it in application logic.
- Treat model output as untrusted input; valid structure does not guarantee factual correctness.
- Keep policy-specific decisions outside the model unless authoritative policy data and appropriate business controls are available.
- Do not let the assistant claim that a claim is approved, rejected, or guaranteed.
- Keep secrets out of source control and logs.

### Completion criteria

The implementation is complete when a customer query can pass through the application, produce a summary, classification, and recommended next steps, and return a consolidated response. The same pipeline must support individual execution, batch processing, and streaming, with explicit handling for invalid inputs and execution failures.
